# Nilons Recommender System 2: For Distributors

## Business Objective

To recommend products to distributors:
1) That sold in the same month last year (seasonality).
2) Based on recent 3-month activity (momentum).
3) Highlighting items that haven't yet sold this month (gap alert).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import xgboost as xgb
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

## Importing Data

The dataset should as a minimum include columns: 
* Distributor id
* Product id
* Invoice Date (use parse_date in import function)
* Sales Value

Optional are columns that contain metadata such as:
* Product category
* Distributor area

In [3]:
prev_df = pd.read_excel(r"D:\OneDrive - Nilons Enterprises Pvt Ltd\Desktop\Anadi\Data\YTD 2024-2025 NC_E.xlsx")
prev_df.head()

,Distribution Channel,Distributor Name,Area,Category,Variants,Invoice Type,Invoice Date,Distributor Code,Item Code,Item Name,Billing Amount
0,EXP,AL GHANIM & AL MAYA GEN. TRAD. CO.,EXPORT,PICKLE-CP,PICKLE-AVAKAI,Invoice,2024-04-12,300007,1500098.0,300 GM NILONS AVAKAI PICKLE PET R/O (ALMAYA EX...,20055.3
1,EXP,AL GHANIM & AL MAYA GEN. TRAD. CO.,EXPORT,PICKLE-CP,PICKLE-CP-TOMATO,Invoice,2024-04-12,300007,1500511.0,300 GM ALMAYA TOMATO PICKLE BTL (1*24),20055.3
2,EXP,AL GHANIM & AL MAYA GEN. TRAD. CO.,EXPORT,PICKLE-CP,PICKLE-MANGO,Invoice,2024-04-12,300007,1500054.0,400 GM KK MANGO PICKLE PET(EXPORT TO KUWAIT)(1...,241605.0
3,EXP,AL GHANIM & AL MAYA GEN. TRAD. CO.,EXPORT,PICKLE-CP,PICKLE-MIX,Invoice,2024-04-12,300007,1500055.0,400 GM KK MIX PICKLE PET(EXPORT TO KUWAIT)(1*12),187915.0
4,EXP,AL GHANIM & AL MAYA GEN. TRAD. CO.,EXPORT,PICKLE-CP,PICKLE-CHILLI,Invoice,2024-04-12,300007,1500058.0,400 GM KK CHILLI PICKLE PET (EXPORT TO KUWAIT)...,107380.0


In [4]:
prev_df.columns = prev_df.columns.str.strip()
prev_df.columns

Index(['Distribution Channel', 'Distributor Name', 'Area', 'Category',
       'Variants', 'Invoice Type', 'Invoice Date', 'Distributor Code',
       'Item Code', 'Item Name', 'Billing Amount'],
      dtype='object')

In [5]:
curr_df = pd.read_excel(r"D:\OneDrive - Nilons Enterprises Pvt Ltd\Desktop\Anadi\Data\SAP25_apr1st_may31st_E.xlsx")
curr_df.head()

,Invoice Type,Invoice Date,Distribution Channel,Material Sub Group,Item Code,Item Name,Bill Amount,C. No,C. Name,C. Area
0,Domestic Invoice,2025-04-10,MT,TOOTY FRUITY,1401079.0,200g*32 CANDIED FRUITY MIX COLOR PET MT,27428.64,2200021.0,AVENUE SUPERMARTS LIMITED-1039,MUMBAI
1,Domestic Invoice,2025-04-10,MT,TOOTY FRUITY,1401080.0,200g*32 CANDIED FRUITY RED COLOR PET MT,14857.18,2200021.0,AVENUE SUPERMARTS LIMITED-1039,MUMBAI
2,Domestic Invoice,2025-04-10,MT,TOOTY FRUITY,1401080.0,200g*32 CANDIED FRUITY RED COLOR PET MT,12571.46,2200021.0,AVENUE SUPERMARTS LIMITED-1039,MUMBAI
3,Domestic Invoice,2025-04-10,MT,SAUCE,1401017.0,660G*24 G-CHILLI SAUCE HDPE JAR MT,14571.40,2200021.0,AVENUE SUPERMARTS LIMITED-1039,MUMBAI
4,Domestic Invoice,2025-04-10,MT,SAUCE,1401018.0,660G*24 R-CHILLI SAUCE HDPE JAR MT,7285.70,2200021.0,AVENUE SUPERMARTS LIMITED-1039,MUMBAI


In [6]:
curr_df.columns = curr_df.columns.str.strip()
curr_df.columns

Index(['Invoice Type', 'Invoice Date', 'Distribution Channel',
       'Material Sub Group', 'Item Code', 'Item Name', 'Bill Amount', 'C. No',
       'C. Name', 'C. Area'],
      dtype='object')

## Data Preprocessing

In [7]:
# Parse dates
prev_df['Invoice Date'] = pd.to_datetime(prev_df['Invoice Date'])
curr_df['Invoice Date'] = pd.to_datetime(curr_df['Invoice Date'])

In [8]:
# Rename key columns for consistency
prev_df.rename(columns={
    'Distributor Code': 'Distributor',
    'Billing Amount': 'Bill Amount'
}, inplace=True)
curr_df.rename(columns={
    'C. No': 'Distributor',
    'C. Name': 'Distributor Name',
    'C. Area': 'Area'
}, inplace=True)

In [9]:
channel_map = {
    'EXP': 'Export',
    'RL': 'Institutional',
    'INST': 'Institutional',
    'GT': 'GT',
    'MT': 'MT',
    'PL': 'Private Label',
    'SMT': 'SMT',
    'GOVT': 'Command',
    'E-COM': 'E-Commerce',
    'GT HO': 'Horeca'
}
prev_df['Distribution Channel'] = prev_df['Distribution Channel'].replace(channel_map)

In [10]:
category_map = {
    'VERMICELLI-ROASTED': 'ROAST VERMICELLI',
    'VERMICELLI-CUT': 'CUT VERMICELLI',
    'TOOTY FRUTI': 'TOOTY FRUITY',
    'RE 1 & 2': 'PICKLE-RE 1&2',
    'BLENDED - WESTERN': 'SPICE-WESTERN BLEND',
    'BLENDED - INDIAN': 'SPICE-INDIAN BLEND',
    'SPICES-BLENDED': 'SPICE-BASIC',
    'SPICES-CTC': 'SPICE-CTC',
    'SPICES-RTC': 'SPICE-RTC',
}
prev_df['Category'] = prev_df['Category'].replace(category_map)

In [11]:
area_map = {
    'ORISSA': 'ODISHA',
    'UTTRAKHAND': 'UTTARAKHAND',
    'BAREILY': 'BAREILLY',
    'PUNE & GOA': 'PUNE',
    'GUJARAT-RAJKOT': 'RAJKOT',
    'GUJARAT-AHMEDABAD': 'AHMEDABAD',
    'CHHATISGARH': 'CHHATTISGARH',
    'BIHAR-MUZAFFARPUR (N)': 'NORTH BIHAR',
    'BIHAR-MUZAFFARPUR (J)': 'SOUTH BIHAR',
    'BIHAR-PATNA': 'NORTH BIHAR',
    'ROM 1': 'ROM',
    'MODERN TRADE': 'MT',
    'PRIVATE LABLE': 'Private Label'
}
prev_df['Area'] = prev_df['Area'].replace(area_map)

## Date Manipulation

In [12]:
# setup reference periods

this_month = pd.to_datetime("2025-05-01")
this_month_str = this_month.strftime('%Y-%m')
last_year_same_month = (this_month.replace(year=this_month.year - 1)).strftime('%Y-%m')
last_3_months = [(this_month.replace(month=m)).strftime('%Y-%m') for m in [2, 3, 4]]

In [13]:
# Add YYYY-MM column to both datasets
prev_df['period'] = prev_df['Invoice Date'].dt.to_period('M').astype(str)
curr_df['period'] = curr_df['Invoice Date'].dt.to_period('M').astype(str)

In [14]:
# Combine both for feature engineering
full_df = pd.concat([prev_df, curr_df], ignore_index=True)

## Create Target Label

In [15]:
# Sold this month (label = 1)
target = curr_df[curr_df['period'] == this_month_str][['Distributor', 'Item Code']].copy()
target['label'] = 1

In [16]:
# Candidates: sold last year same month OR last 3 months
last_year_sales = prev_df[prev_df['period'] == last_year_same_month][['Distributor', 'Item Code']]
recent_sales = curr_df[curr_df['period'].isin(last_3_months)][['Distributor', 'Item Code']]
candidates = pd.concat([last_year_sales, recent_sales]).drop_duplicates()

In [18]:
# Ensure matching data types before merge
target['Distributor'] = target['Distributor'].astype(str)
candidates['Distributor'] = candidates['Distributor'].astype(str)

In [19]:
# Merge with label
data = candidates.merge(target, on=['Distributor', 'Item Code'], how='left')
data['label'] = data['label'].fillna(0)

## Feature Engineering

In [20]:
# Frequency and quantity in last 3 months
freq = curr_df[curr_df['period'].isin(last_3_months)].groupby(['Distributor', 'Item Code']).agg(
    times_sold_last_3=('Invoice Date', 'count'),
    qty_last_3=('Bill Amount', 'sum'),
    last_sold=('Invoice Date', 'max')
).reset_index()

In [21]:
# Seasonal flag from last year
seasonal = prev_df[prev_df['period'] == last_year_same_month][['Distributor', 'Item Code']]
seasonal['sold_last_year_same_month'] = 1

In [24]:
# Merge all features
# Ensure consistent data types for merging
freq['Distributor'] = freq['Distributor'].astype(str)
features = data.merge(freq, on=['Distributor', 'Item Code'], how='left')
features = features.merge(seasonal, on=['Distributor', 'Item Code'], how='left')

features['sold_last_year_same_month'] = features['sold_last_year_same_month'].fillna(0)
features['times_sold_last_3'] = features['times_sold_last_3'].fillna(0)
features['qty_last_3'] = features['qty_last_3'].fillna(0)
features['last_sold'] = pd.to_datetime(features['last_sold'], errors='coerce')
features['days_since_last_sold'] = (this_month - features['last_sold']).dt.days
features['days_since_last_sold'] = features['days_since_last_sold'].fillna(999)

## ML Model

In [25]:
X = features[['sold_last_year_same_month', 'times_sold_last_3', 'qty_last_3', 'days_since_last_sold']]
y = features['label']

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [27]:
model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
model.fit(X_train, y_train)

c:\Users\anadi.k\Github\Nilons-Recommender-System\venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:14:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [28]:
print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

         0.0       0.94      0.88      0.91      4409
         1.0       0.77      0.88      0.82      2004

    accuracy                           0.88      6413
   macro avg       0.86      0.88      0.87      6413
weighted avg       0.89      0.88      0.88      6413



## Results

In [29]:
features['probability'] = model.predict_proba(X)[:, 1]

In [30]:
# Final recommended products: sold last year same month, NOT sold in last 3 months
highlight = features[
    (features['sold_last_year_same_month'] == 1) &
    (features['times_sold_last_3'] == 0)
].copy()
highlight = highlight.sort_values(by='probability', ascending=False)

In [31]:
# Save recommendation output
highlight.to_excel("highlighted_recommendations.xlsx", index=False)

In [32]:
topn = features[features['label'] == 0].sort_values(by='probability', ascending=False)
topn = topn.groupby('Distributor').head(5)
topn.to_excel("topn_ml_recommendations.xlsx", index=False)